In [ ]:
import glob
import pandas as pd
from pathlib import Path
import numpy as np
import os
import pickle
from nilearn.mass_univariate import permuted_ols
import urllib.request

wd = os.getcwd()

In [ ]:
path_ut = (wd[:-4]+'data/')
folder = path_ut + '/ml_metrics_louvain/weighted_cortex_t0.1_o0.8_r1.2'
path_ml_n = (folder+'/*nodal.csv')
path_ml_g = (folder+'/*global.csv')
path_save = (wd[:-4]+'plots')


In [ ]:
modularity_global = pd.read_csv(f'{folder}/modularity_global.csv', header=None)
participation_nodal = pd.read_csv(f'{folder}/participation_mean_nodal.csv', header=None)
flexibility_nodal = pd.read_csv(f'{folder}/flexibility_nodal.csv', header=None)
persistence_global = pd.read_csv(f'{folder}/persistence_global.csv', header=None)
communities = pd.read_csv(f'{folder}/communities_nodal_layer.csv', header=None)

global_df = pd.DataFrame({
    'modularity': modularity_global[0],
    'persistence': persistence_global[0]
})

leb = pd.read_csv(path_ut+'atlas_labels.csv',header=None)
leb.rename(columns={leb.columns[0]: 'regions'}, inplace=True)
leb['regions'] = leb['regions'].str.replace(' ', '', regex=True)
if 'cortex' in folder:
    leb=leb.iloc[:200]

with open(path_ut+"clinic.pkl", "rb") as f:
    cov_df = pickle.load(f)
cov_df = cov_df.drop(columns=["id"]).reset_index(drop=True)

# Fetch the Schaefer parcellation data from the URL
url = 'https://raw.githubusercontent.com/ThomasYeoLab/CBIG/master/stable_projects/brain_parcellation/Schaefer2018_LocalGlobal/Parcellations/MNI/Centroid_coordinates/Schaefer2018_200Parcels_7Networks_order_FSLMNI152_1mm.Centroid_RAS.csv'
response = urllib.request.urlopen(url)
sch = np.genfromtxt(response, delimiter=',')
sch = sch[1:]
coords = sch[:, 2:]
coords=coords.astype(int)

#store labels with coords for later plotting
label_coord = {}
for l ,c in zip(leb['regions'], coords):
    label_coord[l] = c


In [ ]:
commun = communities.values  # (489, 600)

# commun shape: (num_participants, N*L)
N = 200
L = 3
num_participants = commun.shape[0]

# compute number of communities per participant 
n_comms = np.zeros(num_participants)

for p in range(num_participants):
    comm_2d = commun[p].reshape(N, L)
    n_comms[p] = len(np.unique(comm_2d))

#  compute median + MAD
mediani = np.median(n_comms)
mad = np.median(np.abs(n_comms - mediani))
mad_scaled = mad * 1.4826

threshold = mediani + 3 * mad_scaled

#  flag bad participants 
bad = n_comms > threshold

print(f"Median: {mediani}")
print(f"MAD: {mad}")
print(f"Threshold: {threshold}")
print(f"Excluded: {bad.sum()} / {num_participants}")

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.hist(n_comms, bins=40)
plt.axvline(threshold, color='red')
plt.xlabel("Number of communities")
plt.ylabel("Frequency")
plt.title("Distribution of detected communities per participant")
plt.show()

In [ ]:
bad_idx = np.where(bad)[0]
print(bad_idx)

excluded_patients = bad & (cov_df['group'] == 1)
excluded_controls = bad & (cov_df['group'] == 0)

print(excluded_patients.sum(), excluded_controls.sum())

In [ ]:
flexibility_nodal_clean = flexibility_nodal.copy()
flexibility_nodal_clean.loc[bad, :] = np.nan

participation_nodal_clean = participation_nodal.copy()
participation_nodal_clean.loc[bad, :] = np.nan

global_df_clean = global_df.copy()
global_df_clean.loc[bad, :] = np.nan

communities_clean = communities.copy()
communities_clean.loc[bad, :] = np.nan

In [ ]:
num_coms_layer0 = []  # SC layer
num_coms_layer1 = []  # FC layer  
num_coms_layer2 = []  # GM layer

commun_c = communities_clean.values  # (489, 600)
n_comms = np.zeros(num_participants)

for p in range(num_participants):
    comm_2d = commun_c[p].reshape(N, L)
    n_comms[p] = len(np.unique(comm_2d))
    
    # Count unique communities PER LAYER (should be ~10-15 each)
    n_layer0 = len(np.unique(comm_2d[:,0]))  # SC communities
    n_layer1 = len(np.unique(comm_2d[:,1]))  # FC communities  
    n_layer2 = len(np.unique(comm_2d[:,2]))  # GM communities
    
    num_coms_layer0.append(n_layer0)
    num_coms_layer1.append(n_layer1) 
    num_coms_layer2.append(n_layer2)

meani = np.mean(n_comms)
sdi = np.std(n_comms)

print(f"All layer:  {(meani):.1f} ± {(sdi):.1f}")
print(f"SC layer:  {np.mean(num_coms_layer0):.1f} ± {np.std(num_coms_layer0):.1f}")
print(f"FC layer:  {np.mean(num_coms_layer1):.1f} ± {np.std(num_coms_layer1):.1f}")
print(f"GM layer:  {np.mean(num_coms_layer2):.1f} ± {np.std(num_coms_layer2):.1f}")

print(f"Q:  {global_df_clean['modularity'].mean():.2f} ± {global_df_clean['modularity'].std():.2f}")

In [ ]:
### processing functions for statistical analysis:

def zscore_cols(df, cols):
    out = df.copy()
    for c in cols:
        mu = out[c].mean()
        sd = out[c].std(ddof=0)
        out[c] = (out[c] - mu) / (sd if sd != 0 else 1.0)
    return out

def align_data(features_df, covz_df):
    idx = features_df.index.intersection(covz_df.index)
    Y = features_df.loc[idx]
    C = covz_df.loc[idx]
    return Y, C

def build_design_matrices(covz_df):
    # regressor of interest
    X_interest = covz_df[["group"]].astype(float).to_numpy()  # (n, 1)

    # nuisance covariates
    X_nuisance = covz_df[
        [
            "sex",
            "age",
            "d_motion",
            "f_motion",
            "icv",
            "centre_1",
            "centre_2",
            "centre_3",
            ]
            ].astype(float).to_numpy()

    return X_interest, X_nuisance


def run_permutation_test(X_interest, Y, X_nuisance,
                         n_perm=10000, seed=42):
    """
    Y: (n_subjects, n_features)
    """
    out = permuted_ols(
        tested_vars=X_interest, # group, shape (n, 1)
        target_vars=Y, # features, shape (n, p)
        confounding_vars=X_nuisance, # covariates, shape (n, k)
        model_intercept=True,
        n_perm=n_perm, # number of permutations
        two_sided_test=True,
        random_state=seed,
        n_jobs=-1,
        output_type="dict"
    )

    p_fwer = 10 ** (-out["logp_max_t"][0])  # FWER-corrected p-values
    t_vals = out["t"][0]
    null = out["h0_max_t"]#[0]
 
    print(out.keys())

    return p_fwer, t_vals, null

In [ ]:
# perform permutation on global (per column)
covz_df = zscore_cols(cov_df, ["age", "d_motion", "f_motion", "icv"])

centre_dummies = pd.get_dummies(
    covz_df["centre"],
    prefix="centre",
    drop_first=True
).astype(float)

covz_df = pd.concat(
    [covz_df.drop(columns="centre"), centre_dummies],
    axis=1
)


Y, C = align_data(global_df, covz_df)

# remove bad participants
mask = ~bad
Y = Y[mask]
C = C[mask]

X_interest, X_nuisance = build_design_matrices(C)

# run per column 
Y_np = Y.to_numpy()
n_feat = Y_np.shape[1]

p_vals = np.zeros(n_feat)
t_vals = np.zeros(n_feat)

for j in range(n_feat):
    y_col = Y_np[:, j].reshape(-1, 1)

    p, t, _ = run_permutation_test(X_interest, y_col, X_nuisance)

    p_vals[j] = p[0]
    t_vals[j] = t[0]

globals = pd.DataFrame({
    "p_value": p_vals,
    "t_value": t_vals
})

from statsmodels.stats.multitest import fdrcorrection

rej, p_fdr = fdrcorrection(p_vals, alpha=0.05)

globals = pd.DataFrame({
    "t_value": t_vals,
    "p_value": p_vals,
    "p_fdr": p_fdr,
    "significant": rej
})

globals

In [ ]:
# perform permutation on participation (per column)

Y, C = align_data(participation_nodal, covz_df)

# remove bad participants
mask = ~bad
Y = Y[mask]
C = C[mask]

X_interest, X_nuisance = build_design_matrices(C)

# run per node 
Y_np = Y.to_numpy()
n_nodes = Y_np.shape[1]

p_vals = np.zeros(n_nodes)
t_vals = np.zeros(n_nodes)

for j in range(n_nodes):
    y_col = Y_np[:, j].reshape(-1, 1)

    p, t, _ = run_permutation_test(X_interest, y_col, X_nuisance)

    p_vals[j] = p[0]
    t_vals[j] = t[0]

# FDR 
rej, p_fdr = fdrcorrection(p_vals, alpha=0.05)

participation_nodal_p_fdr = pd.DataFrame({
    "regions": leb["regions"],
    "p_value": p_vals,
    "t_value": t_vals,
    "p_fdr": p_fdr,
    "significant": rej
})

# coords (keep your logic)
coord_hits_all = [label_coord[idx] for idx in participation_nodal_p_fdr["regions"] if idx in label_coord]
participation_nodal_p_fdr["Coordinates"] = coord_hits_all

print(participation_nodal_p_fdr[participation_nodal_p_fdr["p_fdr"] < 0.05].sort_values(by="p_fdr"))

In [ ]:
print(participation_nodal_p_fdr.sort_values(by="p_value").head(10))

In [ ]:
# perform permutation on flexibility (per column)

Y, C = align_data(flexibility_nodal, covz_df)

# remove bad participants
mask = ~bad
Y = Y[mask]
C = C[mask]

X_interest, X_nuisance = build_design_matrices(C)

# per node
Y_np = Y.to_numpy()
n_nodes = Y_np.shape[1]

p_vals = np.zeros(n_nodes)
t_vals = np.zeros(n_nodes)

for j in range(n_nodes):
    y_col = Y_np[:, j].reshape(-1, 1)

    p, t, _ = run_permutation_test(X_interest, y_col, X_nuisance)

    p_vals[j] = p[0]
    t_vals[j] = t[0]

# FDR
rej, p_fdr = fdrcorrection(p_vals, alpha=0.05)

flexibility_nodal_p_fdr = pd.DataFrame({
    "regions": leb["regions"],
    "p_value": p_vals,
    "t_value": t_vals,
    "p_fdr": p_fdr,
    "significant": rej
})

coord_hits_all = [label_coord[idx] for idx in flexibility_nodal_p_fdr["regions"] if idx in label_coord]
flexibility_nodal_p_fdr["Coordinates"] = coord_hits_all

# print significant
print(flexibility_nodal_p_fdr[flexibility_nodal_p_fdr["p_fdr"] < 0.05].sort_values(by="p_fdr"))

In [ ]:
flexibility_nodal_ps = flexibility_nodal_p_fdr[flexibility_nodal_p_fdr['p_fdr']<0.05]
print(flexibility_nodal_ps.sort_values(by='p_value'))

In [ ]:
print(round(global_df_clean['modularity'].iloc[:163].mean(), 2),
      round(global_df_clean['modularity'].iloc[:163].std(), 2))

print(round(global_df_clean['modularity'].iloc[163:].mean(), 2),
      round(global_df_clean['modularity'].iloc[163:].std(), 2))

In [ ]:
print(round(global_df_clean['persistence'].iloc[:163].mean(), 2),
      round(global_df_clean['persistence'].iloc[:163].std(), 2))

print(round(global_df_clean['persistence'].iloc[163:].mean(), 2),
      round(global_df_clean['persistence'].iloc[163:].std(), 2))

In [ ]:
# to split the 'Region' string and pulls the network name
def get_network(region_str):
    # Splits 'L_Default_Par_2...' into ['L', 'Default', 'Par', '2'] and takes index 1
    return region_str.split('_')[1]

flexibility_nodal_ps['Network'] = flexibility_nodal_ps['regions'].apply(get_network)

# mapping for the 7-network hierarchy
network_map = {
    'SomMot': 1,
    'Limbic': 2,
    'Cont': 3,
    'Default': 4
}

# create the Network_ID column
flexibility_nodal_ps['Network_ID'] = flexibility_nodal_ps['Network'].map(network_map)

In [ ]:
from nilearn import plotting
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Define the standard Yeo 7-network hex colors - trying to match the Yeo 7-network colors but slightly adjusted for better contast with white background
# Map to Network_ID (0-6)
yeo_colors = [
    "#368ED6", # 1: SomMot (Blue)
    "#DAC400", # 4: Limbic (Cream/Light Green)
    '#E49C1C', # 5: Cont/CEN (Orange)
    "#CC3244"  # 6: Default/DMN (Red) 
]

#Create a discrete colormap from this list
custom_cmap = mcolors.ListedColormap(yeo_colors)

# Setup plotting vars
coordi = np.array(flexibility_nodal_ps["Coordinates"].tolist())
t_values = flexibility_nodal_ps["t_value"].abs() # Use absolute t-value for size
networks = flexibility_nodal_ps["Network_ID"]

# Scale node size by effect size
min_size = 50
max_size = 250
scaled_sizes = min_size + (t_values - t_values.min()) / (t_values.max() - t_values.min()) * (max_size - min_size)

# Plot
fig = plt.figure(figsize=(10, 8))
display = plotting.plot_markers(
    node_values=networks,
    node_coords=coordi,
    node_size=scaled_sizes,
    node_cmap=custom_cmap,
    alpha=0.8,
    display_mode='lzry',
    black_bg=False,
    figure=fig,
    colorbar=False  
)

# legend
from matplotlib.lines import Line2D

legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Sensorimotor', markerfacecolor='#368ED6', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Limbic', markerfacecolor='#DAC400', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Control', markerfacecolor='#E49C1C', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Default Mode', markerfacecolor='#CC3244', markersize=10),
]

plt.legend(handles=legend_elements, loc='lower center', ncol=3, bbox_to_anchor=(-1.2, -0.4))
plt.subplots_adjust(bottom=0.2) 
plt.savefig(path_save + '/flexibility_network_brain.svg', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"

# indices to plot
plot_idx = flexibility_nodal_ps.index.tolist()

vals = [0, 2/3, 1]
labels = ["0", "0.67", "1"]

for node in plot_idx:
    
    # extract region name
    region_name = flexibility_nodal_ps.loc[node, "regions"]
    
    # extract nodal values
    hc = flexibility_nodal_clean.iloc[163:, node].values
    bd = flexibility_nodal_clean.iloc[:163, node].values
    
    # proportions
    hc_prop = [np.nanmean(hc == v) for v in vals]
    bd_prop = [np.nanmean(bd == v) for v in vals]
    
    # grouped bar plot
    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=labels,
        y=hc_prop,
        name="HC",
        text=[f"{p:.2f}" for p in hc_prop],
        textposition="outside",
        marker_color="#4C72B0"
    ))
    
    fig.add_trace(go.Bar(
        x=labels,
        y=bd_prop,
        name="BD",
        text=[f"{p:.2f}" for p in bd_prop],
        textposition="outside",
        marker_color="#C44E52"
    ))

    
    fig.update_layout(
        title={
            "text": region_name.replace("_", " "),
            "x": 0.5,
            "font": dict(size=16),
            "xanchor": "center"
        },
        xaxis_title="Flexibility",
        yaxis_title="Proportion",
        yaxis=dict(range=[0, 1]),
        barmode="group",
        template="simple_white",
        width=600,
        height=450,
        font=dict(size=11),
        legend=dict(
            font=dict(size=11),
            x=0.82,
            y=1
        )
    )
    
    safe_name = region_name.replace(" ", "_").replace("/", "_")
    from kaleido.scopes.plotly import PlotlyScope
    scope = PlotlyScope()
    
    print(f"{node}: {region_name} | mean BD={np.nanmean(bd):.3f}, mean HC={np.nanmean(hc):.3f}")